In [1]:
import pandas as pd
import os
import getpass


from sqlalchemy import create_engine
from sqlalchemy.engine import URL
# Local src

from src.df_overview import df_overview
from src.generate_metadata import generate_metadata
from src.database import create_database
from src.database import create_table
from src.database import load_data
from src.feature_engineering import feature_engineering_sql
from src.feature_engineering import feature_engineering_pandas


# Create database and variables for database connection

db_name = input('Enter database name: ')
password = getpass.getpass('Enter database password: ')
table_name_load = input('Enter table name: ')
table_name_featured = input('Enter table name for featured data: ')

main_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",          # change if needed
    password=password,
    host="localhost",
    port=5432,
    database="postgres"          
)

engine_main = create_engine(main_url)
path_base_csv = '../data/02_processed/base_retail.csv'

print('Done')

Done


In [ ]:
create_database(engine_main, db_name)

created_db_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",         
    password=password,
    host="localhost",
    port=5432,
    database=db_name          
)

engine_created_db = create_engine(created_db_url)

create_table(engine_created_db, table_name_load)

load_data(path_base_csv, engine_created_db, table_name_load)

feature_engineering_sql(engine_created_db, table_name_load, table_name_featured)

# Read the featured table back so df_rfm is available for all downstream cells
df_rfm = pd.read_sql(f"SELECT * FROM public.{table_name_featured}", engine_created_db)
print(f"df_rfm loaded: {df_rfm.shape[0]} rows, {df_rfm.shape[1]} columns")


Database 'retail' created successfully.
Table 'retail_base' created successfully in schema 'public'.
Data loaded successfully into table 'retail_base'.
Feature engineering table public.retail_featured created successfully.


In [3]:
'''
pandas feature engineering for testing purposes.
if you have any issues with the SQL feature engineering, you can use pandas version for feature engineering, 
but comment out the SQL version and further testing code to avoid confusion.

df = pd.read_csv(path_base_csv)
df['invoice_date'] = pd.to_datetime(df['invoice_date'])
df_rfm = feature_engineering_pandas(df)
'''

"\npandas feature engineering for testing purposes.\nif you have any issues with the SQL feature engineering, you can use pandas version for feature engineering, \nbut comment out the SQL version and further testing code to avoid confusion.\n\ndf = pd.read_csv(path_base_csv)\ndf['invoice_date'] = pd.to_datetime(df['invoice_date'])\ndf_rfm = feature_engineering_pandas(df)\n"

In [4]:
df_overview(df_rfm)

NameError: name 'df_rfm' is not defined

In [ ]:
pd_to_csv_path = os.path.join('..', 'data', '03_featured', 'featured_rfm_retail.csv')

os.makedirs(os.path.dirname(pd_to_csv_path), exist_ok=True)
df_rfm.to_csv(pd_to_csv_path)

generate_metadata(
    df=df_rfm,
    csv_path=pd_to_csv_path
)

print(f"Saved to: {pd_to_csv_path}")

Metadata saved: ..\data\03_featured\featured_rfm_retail_metadata.json
Saved to: ..\data\03_featured\featured_rfm_retail.csv
